In [1]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np
from rasterio.transform import from_origin

# Input shapefile (in lat/lon)
shapefile = "D:\Phd Research\GIS\Shape\land_Part_area_utm.zip"
gdf = gpd.read_file(f"zip://{shapefile}")

# Reproject to UTM for metric raster (1 m resolution)
gdf = gdf.to_crs(epsg=32614)  # UTM Zone 14N for Texas

# Get bounds and resolution
minx, miny, maxx, maxy = gdf.total_bounds
res = 200  # 200 meter resolution
width = int((maxx - minx) / res)
height = int((maxy - miny) / res)

# Create a uniform raster with constant value 6.5
array = np.full((height, width), 7.5, dtype=np.float32)

# Define transform
transform = from_origin(minx, maxy, res, res)

# Save temporary full raster
temp_raster = "uniform_full.tif"
with rasterio.open(
    temp_raster,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="float32",
    crs=gdf.crs,
    transform=transform,
) as dst:
    dst.write(array, 1)

# Mask raster with shapefile
with rasterio.open(temp_raster) as src:
    out_image, out_transform = mask(src, gdf.geometry, crop=True, filled=True)
    out_meta = src.meta.copy()
    out_meta.update({
        "height": out_image.shape[1],
        "width": out_image.shape[2],
        "transform": out_transform
    })

# Save the masked output
masked_raster = "uniform_surge_SLR_100yr_7.5.tif"
with rasterio.open(masked_raster, "w", **out_meta) as dest:
    dest.write(out_image)

print(f"✅ Uniform masked raster saved to: {masked_raster}")


✅ Uniform masked raster saved to: uniform_surge_SLR_100yr_7.5.tif


In [5]:
# pip install -U mikeio1d pandas
import mikeio1d, pandas as pd, inspect

FN = r"D:\Phd Research\1D HD Model\100 yr CC wirhout reef_m1d - Result Files\texas_coast_1BaseDefault_Network_HD.res1d"

res = mikeio1d.open(FN)

print("\n=== BASIC INFO ===")
print("mikeio1d version:", getattr(mikeio1d, "__version__", "?"))
print("Reaches:", getattr(res.reaches, "names", []))
print("Nodes  :", len(getattr(res.nodes, "names", [])))

# 1) See what attributes exist that might list quantities/items
print("\n=== POSSIBLE ITEM LISTS ON res ===")
for name in ("items", "item_names", "quantities", "result_types"):
    val = getattr(res, name, None)
    if val is None:
        continue
    try:
        shown = list(val) if not isinstance(val, dict) else list(val.keys())
        print(f"{name}: {shown[:12]}{' ...' if len(shown)>12 else ''}")
    except Exception:
        print(f"{name}: <present but not list-like>")

# 2) Inspect the read() signature to see accepted keyword names
print("\n=== read() SIGNATURE ===")
try:
    sig = inspect.signature(res.read)
    print("res.read", sig)
except Exception as e:
    print("Could not inspect res.read signature:", e)

# 3) Try common keyword names x common WL labels; report what works
kw_candidates = ["item", "items", "quantity", "quantities"]
wl_labels = [
    "Water level", "Water Level", "ExternalWaterLevel",
    "Stage", "Head", "H", "WL"
]

print("\n=== PROBE GLOBAL res.read(...) ===")
successes = []
for kw in kw_candidates:
    for lbl in wl_labels:
        try:
            ds = res.read(**{kw: lbl})
            df = ds.to_dataframe()
            if df.shape[1] > 0:
                successes.append((kw, lbl, df.shape))
                print(f"OK: res.read({kw}={lbl!r}) -> shape={df.shape}; sample cols: {list(df.columns)[:5]}")
        except TypeError:
            # keyword not accepted by this build
            pass
        except Exception:
            # accepted kw, but label not present/usable
            pass
if not successes:
    print("No success via global res.read(...). Will try per-reach groups next.")

# 4) Per-reach discovery: find attributes that have .read() and look like water level
print("\n=== PROBE PER-REACH GROUPS ===")
reach_hits = []
for rname in getattr(res.reaches, "names", [])[:5]:  # check first few reaches
    reach = res.reaches[rname]
    attrs = [a for a in dir(reach) if not a.startswith("_")]
    # candidates that expose .read()
    cand = []
    for a in attrs:
        obj = getattr(reach, a, None)
        if hasattr(obj, "read"):
            if any(k in a.lower() for k in ("level", "stage", "head", "water")):
                cand.append(a)
    if cand:
        print(f"{rname}: candidate groups -> {cand}")
        # try each candidate group
        for a in cand:
            try:
                df = getattr(reach, a).read()
                if df is not None and df.shape[1] > 0:
                    reach_hits.append((rname, a, df.shape))
                    print(f"OK: reach[{rname!r}].{a}.read() -> shape={df.shape}; sample cols: {list(df.columns)[:5]}")
            except Exception:
                pass

if not reach_hits and not successes:
    print("\nNo readable WL found programmatically. If MIKE View shows WL, double-check:")
    print("  • Python points to the SAME .res1d path MIKE View opened.")
    print("  • Try Python 3.10 and `pip install -U mikeio1d` if you’re on 3.11.")
    print("  • MIKE+ is installed on this machine; restart Python after installation.\n")

# 5) If something worked, print the exact call you should use later
print("\n=== SUMMARY ===")
if successes:
    kw, lbl, shp = successes[0]
    print(f"Use: res.read({kw}={lbl!r})  -> dataset shape {shp}")
elif reach_hits:
    rname, a, shp = reach_hits[0]
    print(f"Use per-reach: res.reaches[{rname!r}].{a}.read()  -> dataframe shape {shp}")
else:
    print("Nothing worked via probes. Re-check the three bullets above.")



=== BASIC INFO ===
mikeio1d version: 1.1.1
Reaches: ['PLACEDO_CK', 'COLORODO_RV', 'ARANSAS_RV', 'W_MUSTANG_CK', 'E_MUSTANG_CK', 'LAVACA_RV', 'GARCITAS_CK', 'LOS_OLMOS_CK', 'OSO_CK', 'TRANQUITAS_CK', 'NUICES_RV', 'GAUDALUPE_RV', 'PATRONILA_CK', 'MISSION_RV', 'COPANO_CK', 'SANDY_CK', 'TRES_PLACIOS_R']
Nodes  : 34

=== POSSIBLE ITEM LISTS ON res ===
quantities: ['DischargeToSurface', 'ExternalWaterLevel', 'LeftLatLinkOutflow', 'RightLatLinkOutflow']

=== read() SIGNATURE ===
res.read (queries: 'Optional[list[TimeSeriesId] | TimeSeriesId | list[QueryData] | QueryData]' = None, column_mode: 'Optional[str | ColumnMode]' = None) -> 'pd.DataFrame'

=== PROBE GLOBAL res.read(...) ===
No success via global res.read(...). Will try per-reach groups next.

=== PROBE PER-REACH GROUPS ===

No readable WL found programmatically. If MIKE View shows WL, double-check:
  • Python points to the SAME .res1d path MIKE View opened.
  • Try Python 3.10 and `pip install -U mikeio1d` if you’re on 3.11.
  • MIKE